# Recipe 2: Decision Trees and Random Forests
**EDPY 506 · Machine Learning: Theory and Applications · Recipe notebooks**

<a href="https://colab.research.google.com/github/okanbulut/edpy506/blob/main/recipes/notebooks/Recipe02_Trees_Forests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A **decision tree** asks a series of yes/no questions ("Did the student click more than 300 times?") and ends in a prediction. It is easy to read but can memorize noise. A **random forest** grows hundreds of slightly different trees and lets them vote, which is usually far more accurate.

**Lectures:** 2 (Regression: trees and forests) and 4 (Feature selection and tuning)
**Data:** Open University Learning Analytics Dataset (OULAD; Kuzilek, Hlosta, & Zdrahal, 2017), the same file as Assignment 2: 2,237 students in one social sciences module (BBB, 2013J), with their clicks on the module website, background information, and final result.
**Runtime:** about one minute on a regular (CPU) runtime. No setup or uploads are needed: just run the cells from top to bottom (**Runtime → Run all**).

**How this notebook is organized.** It follows the seven steps of the *ML Workflow Recipe* paper:
1. Define the problem · 2. Split the data · 3. Preprocess · 4. Baseline · 5. Train and tune · 6. Evaluate · 7. Interpret

After each output there is a **What you should see** note. Your numbers should match closely if you keep the default parameters. If you change the parameters, the numbers will change, and that is the point: try it and see what happens.

In [ ]:
# Run this cell first. It loads the tools we use.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.precision", 2)
plt.rcParams["figure.dpi"] = 110

## ✏️ Change these parameters
This is the **only cell you need to edit**. Change a value, then choose **Runtime → Run all** to rerun the whole recipe.

In [ ]:
# ===== CHANGE THESE PARAMETERS =====
TEST_SIZE = 0.2      # share of students kept aside for the final test
SEED = 506           # keeps the random parts the same every run
MAX_DEPTH = 3        # how many questions deep the single tree may go (try 2, 5, 10, None)
N_TREES = 300        # number of trees in the random forest (try 10, 100, 500)
MIN_LEAF = 10        # smallest group of students allowed at the end of a branch
# ====================================

In [ ]:
# @title Load the OULAD data (just run this cell; you do not need to read it)
# The data file is stored inside this notebook, so there is nothing to upload.
# If a file called oulad_bbb2013j.csv is in the Files panel, that copy is used instead.
import base64, gzip, io, os
OULAD_GZ = "..."
import pandas as pd
if os.path.exists("oulad_bbb2013j.csv"):
    df = pd.read_csv("oulad_bbb2013j.csv")
else:
    df = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(OULAD_GZ))))
print(f"Loaded {df.shape[0]:,} students and {df.shape[1]} columns.")

## Step 1. Define the problem
Same question as Recipe 1, Part B: among students who stayed to the end, who **passed** and who **failed**? Using the same question lets you compare methods fairly.

In [ ]:
# Website engagement (counted over the whole module) and background information
ENGAGEMENT = ["clicks_content", "clicks_resource", "clicks_subpage", "clicks_url", "clicks_forum",
              "clicks_quiz", "clicks_homepage", "active_days", "n_materials", "prestart_clicks"]
BACKGROUND_NUM = ["num_of_prev_attempts", "studied_credits", "registration_day"]
BACKGROUND_CAT = ["gender", "age_band", "highest_education", "imd_band", "disability"]

# Keep the students who stayed to the end of the module (drop "Withdrawn")
stayed = df[df["final_result"] != "Withdrawn"].copy()
stayed["passed"] = stayed["final_result"].isin(["Pass", "Distinction"]).astype(int)   # 1 = passed, 0 = failed

num_cols = ENGAGEMENT + BACKGROUND_NUM
cat_cols = BACKGROUND_CAT
X = stayed[num_cols + cat_cols]
y = stayed["passed"]
print(f"{len(stayed):,} students stayed to the end; {y.mean():.0%} of them passed.")

> **What you should see:** The same 1,593 students and the same 67% pass rate as Recipe 1.

## Step 2. Split the data

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f"Training: {len(X_train):,}   Test: {len(X_test):,}")

## Step 3. Preprocess
Trees do **not** need scaling: a question like "more than 300 clicks?" works the same whatever the units. They still need categories turned into numbers, so we one-hot encode those.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
preprocess = ColumnTransformer([
    ("numbers", SimpleImputer(strategy="median"), num_cols),
    ("categories", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                 OneHotEncoder(handle_unknown="ignore", sparse_output=False)), cat_cols),
], verbose_feature_names_out=False)

## Step 4. Baselines
Two baselines: guess the most common answer, and the logistic regression from Recipe 1.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

results = {}
def evaluate(name, model):
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    results[name] = {"train accuracy": accuracy_score(y_train, model.predict(X_train)),
                     "test accuracy": accuracy_score(y_test, model.predict(X_test)),
                     "test ROC AUC": roc_auc_score(y_test, p)}
    return model

evaluate("Baseline (most common)", DummyClassifier(strategy="most_frequent"))
evaluate("Logistic regression", Pipeline([("prep", preprocess), ("scale", StandardScaler()),
                                          ("model", LogisticRegression(max_iter=2000))]))
pd.DataFrame(results).T

> **What you should see:** The baseline gets 67% accuracy and a ROC AUC of 0.50 (no better than chance). Logistic regression gets about 86% accuracy and a ROC AUC of about 0.88. These are the numbers the trees must beat.

## Step 5. Train a single tree, then a forest
`MAX_DEPTH` limits how many questions the tree may ask. A shallow tree is easy to read; a very deep tree memorizes the training students.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
tree = evaluate(f"Decision tree (depth {MAX_DEPTH})",
                Pipeline([("prep", preprocess),
                          ("model", DecisionTreeClassifier(max_depth=MAX_DEPTH, min_samples_leaf=MIN_LEAF, random_state=SEED))]))

plt.figure(figsize=(16, 7))
plot_tree(tree.named_steps["model"], feature_names=tree.named_steps["prep"].get_feature_names_out(),
          class_names=["failed", "passed"], filled=True, rounded=True, fontsize=10, max_depth=3, impurity=False, proportion=True)
plt.title("The decision tree (top 3 levels)"); plt.show()

> **What you should see:** A tree with three levels of questions. The first question is about **quiz clicks** (about 50 clicks splits the students), and the next ones are about content clicks and active days. Orange boxes predict "failed" and blue boxes "passed"; darker colour means a purer group. Each box shows the share of students in it and the split between the two classes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = evaluate(f"Random forest ({N_TREES} trees)",
                  Pipeline([("prep", preprocess),
                            ("model", RandomForestClassifier(n_estimators=N_TREES, min_samples_leaf=MIN_LEAF,
                                                             random_state=SEED, n_jobs=-1))]))
print("Forest trained.")

## Step 6. Evaluate
Compare **training** and **test** accuracy. A big gap means the model memorized the training students (*overfitting*).

In [ ]:
table = pd.DataFrame(results).T
print(table.round(3))
table[["train accuracy", "test accuracy"]].plot.barh(color=["#9AA3C7", "#4E5089"], figsize=(7, 3.5), xlim=(0.5, 1))
plt.title("Training vs. test accuracy"); plt.show()

> **What you should see:** All three real models land close together: about 85–86% test accuracy and ROC AUC of 0.88–0.90. The forest has the best ROC AUC (about 0.90). The gaps between training and test accuracy are small (2–3 points) because `MAX_DEPTH` and `MIN_LEAF` keep the models from memorizing.

### How deep should the tree be?
The loop below grows trees of depth 1 to 15 and plots training and test accuracy. This is the **bias–variance trade-off** from Lecture 2 in one picture.

In [ ]:
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    m = Pipeline([("prep", preprocess), ("model", DecisionTreeClassifier(max_depth=d, random_state=SEED))]).fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, m.predict(X_train)))
    test_acc.append(accuracy_score(y_test, m.predict(X_test)))
plt.plot(depths, train_acc, "o-", color="#9AA3C7", label="training")
plt.plot(depths, test_acc, "s-", color="#4E5089", label="test")
plt.xlabel("Maximum depth"); plt.ylabel("Accuracy"); plt.legend(); plt.title("Deeper is not always better")
plt.show()

> **What you should see:** Training accuracy (light line) climbs to nearly 100% as the tree gets deeper, but test accuracy (dark line) **falls**, from about 88% at depth 1 to about 77% at depth 15. Deep trees memorize the training students. Here, one or two questions already do most of the work.

## Step 7. Interpret
A forest has no single set of questions to read, but it can tell us which variables it relied on most. **Permutation importance** shuffles one column at a time in the *test* data and measures how much accuracy drops. A big drop means the model leaned on that variable.

In [ ]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=SEED, n_jobs=-1)
imp = pd.Series(100 * imp.importances_mean, index=X_test.columns).sort_values()
imp.tail(10).plot.barh(color="#4E5089", title="Permutation importance")
plt.xlabel("drop in test accuracy (percentage points)"); plt.show()
imp.sort_values(ascending=False).head(5).round(1).to_frame("drop in accuracy (points)")

> **What you should see:** Quiz clicks matter most (shuffling them costs about 4.6 points of accuracy), then content clicks (about 2.7 points). Everything else costs less than 1 point. Note that the forest can use a variable that is correlated with another and still show low importance for both, because shuffling one leaves the other to stand in.

## Try this next
1. Set `MAX_DEPTH = None` (no limit) and `MIN_LEAF = 1`. Training accuracy hits 100%. What happens to test accuracy?
2. Set `N_TREES = 10`, then `500`. Does the forest keep improving, or does it level off?
3. Recipe 3 shows how to choose `MAX_DEPTH`, `MIN_LEAF`, and other settings properly, with cross-validation instead of the test set.

---
## Where to go next
- **Companion paper:** *Companion papers/Recipe02_Trees_Forests_Companion.pdf* explains the idea behind this recipe in plain words.
- **Workflow:** *01_ML_Workflow_Recipe.pdf* has a checklist for every step used here.
- **Which method?** *02_Which_Method_Flowchart.pdf* helps you pick the right recipe for a new question.
- **Next recipe:** Recipe 3 (regularization and tuning) shows how to choose settings such as `MAX_DEPTH` without peeking at the test set.

*Data citation:* Kuzilek, J., Hlosta, M., & Zdrahal, Z. (2017). Open University Learning Analytics dataset. *Scientific Data, 4*, 170171. https://doi.org/10.1038/sdata.2017.171